In [2]:
import os, json, tarfile, shutil
from collections import defaultdict

SHARDS_DIR   = "/home/rishabh/Downloads/umi-pipeline-training/shards"
FILTERED_DIR = "/home/rishabh/Downloads/umi-pipeline-training/shards_marker_only"
RDT2_DIR     = "/home/rishabh/Downloads/umi-pipeline-training/RDT2"

# From instructions.json:
# task_0 = 'pick up the marker and place it in the box'  ← WANT THIS
# task_1 = 'grasp the cup...'
# task_2 = 'fold the cloth...'
# task_3 = 'open the drawer...'
# task_4 = 'stack the blocks...'

TARGET_KEY = "task_0"  # ← This is the marker task!

print("=" * 60)
print(f"Target key: '{TARGET_KEY}'")
print(f"= 'pick up the marker and place it in the box'")
print("=" * 60)

shard_files = sorted([
    f for f in os.listdir(SHARDS_DIR)
    if f.endswith('.tar')
])

print(f"\nTotal shards: {len(shard_files)}")
print("Scanning shards for task_0...")
print("-" * 50)

marker_shards = []
task_dist     = defaultdict(int)

for i, shard_name in enumerate(shard_files):
    shard_path = os.path.join(SHARDS_DIR, shard_name)
    
    try:
        with tarfile.open(shard_path, 'r') as tar:
            members   = tar.getnames()
            meta_files = [m for m in members if 'meta.json' in m]
            
            if not meta_files:
                continue
            
            # Read FIRST meta.json in shard
            f    = tar.extractfile(meta_files[0])
            meta = json.load(f)
            
            # Get sub_task_instruction_key
            task_key = meta.get('sub_task_instruction_key', 'unknown')
            task_dist[task_key] += 1
            
            # Check if this is marker task
            if task_key == TARGET_KEY:
                marker_shards.append(shard_name)
                
    except Exception as e:
        pass
    
    if (i + 1) % 100 == 0:
        print(f"  Processed {i+1}/{len(shard_files)}... "
              f"marker={len(marker_shards)}")

print("\n" + "=" * 60)
print("TASK DISTRIBUTION:")
print("=" * 60)
for task_key, count in sorted(task_dist.items()):
    tag = "← TARGET (marker)" if task_key == TARGET_KEY else ""
    print(f"  {task_key}: {count} shards  {tag}")

print(f"\n✅ Marker shards found: {len(marker_shards)}")
print(f"   First 5: {marker_shards[:5]}")
print(f"   Last  5: {marker_shards[-5:]}")

Target key: 'task_0'
= 'pick up the marker and place it in the box'

Total shards: 740
Scanning shards for task_0...
--------------------------------------------------
  Processed 100/740... marker=100
  Processed 200/740... marker=200
  Processed 300/740... marker=300
  Processed 400/740... marker=400
  Processed 500/740... marker=500
  Processed 600/740... marker=600
  Processed 700/740... marker=700

TASK DISTRIBUTION:
  task_0: 740 shards  ← TARGET (marker)

✅ Marker shards found: 740
   First 5: ['shard-000000.tar', 'shard-000001.tar', 'shard-000002.tar', 'shard-000003.tar', 'shard-000004.tar']
   Last  5: ['shard-000735.tar', 'shard-000736.tar', 'shard-000737.tar', 'shard-000738.tar', 'shard-000739.tar']


In [3]:
# Clean old filtered dir and create fresh one
if os.path.exists(FILTERED_DIR):
    shutil.rmtree(FILTERED_DIR)
    print(f"🗑️  Removed old: {FILTERED_DIR}")
    
os.makedirs(FILTERED_DIR, exist_ok=True)

print(f"\n✅ Found {len(marker_shards)} marker-only shards")
print(f"Copying to: {FILTERED_DIR}")
print("-" * 50)

for i, shard_name in enumerate(marker_shards):
    src = os.path.join(SHARDS_DIR, shard_name)
    dst = os.path.join(FILTERED_DIR, shard_name)
    shutil.copy2(src, dst)
    
    if (i + 1) % 25 == 0:
        print(f"  Copied {i+1}/{len(marker_shards)}...")

# Create instructions.json with ONLY marker task
instructions = {
    "task_0": "pick up the marker and place it in the box"
}
with open(os.path.join(FILTERED_DIR, 'instructions.json'), 'w') as f:
    json.dump(instructions, f, indent=2)

print(f"\n✅ Copied {len(marker_shards)} shards")
print(f"✅ Created instructions.json")

# Verify contents
verify_shards = sorted([
    f for f in os.listdir(FILTERED_DIR)
    if f.endswith('.tar')
])
print(f"✅ Verified: {len(verify_shards)} .tar files in filtered dir")

# Show sample
print(f"\nSample shards: {verify_shards[:5]}")

🗑️  Removed old: /home/rishabh/Downloads/umi-pipeline-training/shards_marker_only

✅ Found 740 marker-only shards
Copying to: /home/rishabh/Downloads/umi-pipeline-training/shards_marker_only
--------------------------------------------------
  Copied 25/740...
  Copied 50/740...
  Copied 75/740...
  Copied 100/740...
  Copied 125/740...
  Copied 150/740...
  Copied 175/740...
  Copied 200/740...
  Copied 225/740...
  Copied 250/740...
  Copied 275/740...
  Copied 300/740...
  Copied 325/740...
  Copied 350/740...
  Copied 375/740...
  Copied 400/740...
  Copied 425/740...
  Copied 450/740...
  Copied 475/740...
  Copied 500/740...
  Copied 525/740...
  Copied 550/740...
  Copied 575/740...
  Copied 600/740...
  Copied 625/740...
  Copied 650/740...
  Copied 675/740...
  Copied 700/740...
  Copied 725/740...

✅ Copied 740 shards
✅ Created instructions.json
✅ Verified: 740 .tar files in filtered dir

Sample shards: ['shard-000000.tar', 'shard-000001.tar', 'shard-000002.tar', 'shard-00000

In [4]:
# Create dataset config pointing to filtered data
config = f"""name: custom/robot_dataset
type: single
shards_dir: {FILTERED_DIR}
kwargs:
  instruction_path: {FILTERED_DIR}/instructions.json
  normalizer_path: /home/rishabh/Downloads/umi-pipeline-training/umi_normalizer_official.pt
"""

config_path = os.path.join(
    RDT2_DIR,
    "configs/datasets/marker_only_dataset.yaml"
)

with open(config_path, 'w') as f:
    f.write(config)

print("✅ Dataset config created!")
print(f"   Path: {config_path}")
print(f"\n📋 Config contents:")
print(config)

# Also verify instructions.json
with open(os.path.join(FILTERED_DIR, 'instructions.json')) as f:
    instr = json.load(f)
print(f"📋 Instructions: {instr}")

✅ Dataset config created!
   Path: /home/rishabh/Downloads/umi-pipeline-training/RDT2/configs/datasets/marker_only_dataset.yaml

📋 Config contents:
name: custom/robot_dataset
type: single
shards_dir: /home/rishabh/Downloads/umi-pipeline-training/shards_marker_only
kwargs:
  instruction_path: /home/rishabh/Downloads/umi-pipeline-training/shards_marker_only/instructions.json
  normalizer_path: /home/rishabh/Downloads/umi-pipeline-training/umi_normalizer_official.pt

📋 Instructions: {'task_0': 'pick up the marker and place it in the box'}


In [5]:
# Quick sanity check - verify filtered shards are all task_0
print("🔍 Verifying 5 random filtered shards...")
print("-" * 50)

import random
sample = random.sample(verify_shards, min(5, len(verify_shards)))

all_correct = True
for shard_name in sample:
    shard_path = os.path.join(FILTERED_DIR, shard_name)
    with tarfile.open(shard_path, 'r') as tar:
        meta_files = [m for m in tar.getnames() if 'meta.json' in m]
        f    = tar.extractfile(meta_files[0])
        meta = json.load(f)
        key  = meta.get('sub_task_instruction_key', 'unknown')
        ok   = key == TARGET_KEY
        if not ok:
            all_correct = False
        print(f"  {shard_name}: key='{key}' {'✅' if ok else '❌'}")

print(f"\n{'✅ All shards verified as task_0!' if all_correct else '❌ Some shards have wrong task!'}")

print("\n" + "=" * 60)
print("📊 FILTERED DATASET SUMMARY")
print("=" * 60)
print(f"  Total shards:      {len(shard_files)} (original)")
print(f"  Marker shards:     {len(marker_shards)} (task_0 only)")
print(f"  Reduction:         {len(shard_files)-len(marker_shards)} shards removed")
print(f"  Task:              'pick up the marker and place it in the box'")
print(f"  Normalizer:        umi_normalizer_official.pt")
print(f"  Config:            {config_path}")
print("=" * 60)
print("\n✅ Ready to train! Run Cell 5 to start training.")

🔍 Verifying 5 random filtered shards...
--------------------------------------------------
  shard-000032.tar: key='task_0' ✅
  shard-000553.tar: key='task_0' ✅
  shard-000303.tar: key='task_0' ✅
  shard-000228.tar: key='task_0' ✅
  shard-000375.tar: key='task_0' ✅

✅ All shards verified as task_0!

📊 FILTERED DATASET SUMMARY
  Total shards:      740 (original)
  Marker shards:     740 (task_0 only)
  Reduction:         0 shards removed
  Task:              'pick up the marker and place it in the box'
  Normalizer:        umi_normalizer_official.pt
  Config:            /home/rishabh/Downloads/umi-pipeline-training/RDT2/configs/datasets/marker_only_dataset.yaml

✅ Ready to train! Run Cell 5 to start training.


In [9]:
# CELL 4.6 - Generate Normalizer FROM YOUR DATA
import os, json, tarfile, torch, numpy as np
from models.normalizer.normalizer import LinearNormalizer, SingleFieldLinearNormalizer

BASE_DIR        = "/home/rishabh/Downloads/umi-pipeline-training"
FILTERED_DIR    = f"{BASE_DIR}/shards_marker_only"
RDT2_DIR        = f"{BASE_DIR}/RDT2"
NORMALIZER_PATH = f"{BASE_DIR}/umi_normalizer_official.pt"

os.chdir(RDT2_DIR)  # needed for the import above to work

# ── Step 1: Collect all actions from your filtered shards ────
print("📊 Scanning filtered shards to compute action statistics...")
print(f"   Reading from: {FILTERED_DIR}\n")

all_actions = []
shard_files = sorted([
    f for f in os.listdir(FILTERED_DIR) if f.endswith('.tar')
])

for i, shard_name in enumerate(shard_files):
    shard_path = os.path.join(FILTERED_DIR, shard_name)
    try:
        with tarfile.open(shard_path, 'r') as tar:
            members = tar.getnames()
            action_files = [m for m in members if m.endswith('action.npy')]
            for af in action_files:
                f = tar.extractfile(af)
                if f:
                    action = np.load(f)  # shape: (24, 20)
                    all_actions.append(action)
    except Exception as e:
        pass
    
    if (i + 1) % 100 == 0:
        print(f"   Scanned {i+1}/{len(shard_files)} shards, "
              f"collected {len(all_actions)} episodes...")

print(f"\n✅ Collected {len(all_actions)} action episodes")

# ── Step 2: Compute statistics ───────────────────────────────
all_actions = np.stack(all_actions)  # (N, 24, 20)
all_flat    = all_actions.reshape(-1, 20)  # (N*24, 20)

act_min  = all_flat.min(axis=0).astype(np.float32)   # (20,)
act_max  = all_flat.max(axis=0).astype(np.float32)
act_mean = all_flat.mean(axis=0).astype(np.float32)
act_std  = all_flat.std(axis=0).astype(np.float32)
act_std  = np.where(act_std < 1e-6, 1.0, act_std)   # avoid div-by-zero

print(f"\n📈 Action statistics (20-dim):")
print(f"   min range:  [{act_min.min():.4f}, {act_min.max():.4f}]")
print(f"   max range:  [{act_max.min():.4f}, {act_max.max():.4f}]")
print(f"   mean range: [{act_mean.min():.4f}, {act_mean.max():.4f}]")
print(f"   std  range: [{act_std.min():.4f}, {act_std.max():.4f}]")

# ── Step 3: Build LinearNormalizer (limits mode → scale to [-1,1]) 
scale  = 2.0 / (act_max - act_min + 1e-6)       # (20,)
offset = -1.0 - scale * act_min                  # (20,)

normalizer = LinearNormalizer()
normalizer.params_dict = {
    "action": {
        "scale":  torch.from_numpy(scale),
        "offset": torch.from_numpy(offset),
        "input_stats": {
            "min":  torch.from_numpy(act_min),
            "max":  torch.from_numpy(act_max),
            "mean": torch.from_numpy(act_mean),
            "std":  torch.from_numpy(act_std),
        }
    }
}

# ── Step 4: Save it ──────────────────────────────────────────
torch.save(normalizer.params_dict, NORMALIZER_PATH)
size_mb = os.path.getsize(NORMALIZER_PATH) / 1e6
print(f"\n✅ Normalizer saved: {NORMALIZER_PATH}  ({size_mb:.3f} MB)")

# ── Step 5: Verify it loads correctly ────────────────────────
test = LinearNormalizer.load(NORMALIZER_PATH)
test_input  = torch.zeros(1, 24, 20)
test_output = test["action"].normalize(test_input)
print(f"✅ Verify load+normalize: input zeros → output {test_output[0,0,:3].tolist()}")

# ── Step 6: Update config ─────────────────────────────────────
config = f"""name: custom/robot_dataset
type: single
shards_dir: {FILTERED_DIR}
kwargs:
  instruction_path: {FILTERED_DIR}/instructions.json
  normalizer_path: {NORMALIZER_PATH}
"""
config_path = f"{RDT2_DIR}/configs/datasets/marker_only_dataset.yaml"
with open(config_path, 'w') as f:
    f.write(config)

print(f"\n✅ Config updated: {config_path}")
print("\n🚀 Ready — run Cell 5 to start training!")

📊 Scanning filtered shards to compute action statistics...
   Reading from: /home/rishabh/Downloads/umi-pipeline-training/shards_marker_only

   Scanned 100/740 shards, collected 0 episodes...
   Scanned 200/740 shards, collected 0 episodes...
   Scanned 300/740 shards, collected 0 episodes...
   Scanned 400/740 shards, collected 0 episodes...
   Scanned 500/740 shards, collected 0 episodes...
   Scanned 600/740 shards, collected 0 episodes...
   Scanned 700/740 shards, collected 0 episodes...

✅ Collected 0 action episodes


ValueError: need at least one array to stack

In [7]:
# CELL 4.5 - Find & Fix Normalizer
import os, glob

base = "/home/rishabh/Downloads/umi-pipeline-training"

print("=" * 60)
print("🔍 SEARCHING FOR NORMALIZER FILES")
print("=" * 60)

# Search for any .pt files
pt_files = glob.glob(f"{base}/**/*.pt", recursive=True)

if pt_files:
    print(f"\nFound {len(pt_files)} .pt file(s):")
    for f in pt_files:
        size_mb = os.path.getsize(f) / 1e6
        print(f"  {f}")
        print(f"  └─ {size_mb:.2f} MB")
else:
    print("\n❌ No .pt files found anywhere in the directory!")
    print("   You need to download the normalizer.")

# Also check if it's named slightly differently
print("\n🔍 Searching for anything with 'normaliz' in name...")
for root, dirs, files in os.walk(base):
    # Skip hidden dirs and large shard dirs to be fast
    dirs[:] = [d for d in dirs if not d.startswith('.') 
               and 'shards' not in d]
    for f in files:
        if 'normaliz' in f.lower():
            full = os.path.join(root, f)
            size_mb = os.path.getsize(full) / 1e6
            print(f"  FOUND: {full}  ({size_mb:.2f} MB)")

🔍 SEARCHING FOR NORMALIZER FILES

Found 7 .pt file(s):
  /home/rishabh/Downloads/umi-pipeline-training/RDT2/C:\Users\YourName\datasets\normalizer.pt
  └─ 0.00 MB
  /home/rishabh/Downloads/umi-pipeline-training/outputs/rdt2-finetuned/checkpoint-5000/optimizer.pt
  └─ 334.58 MB
  /home/rishabh/Downloads/umi-pipeline-training/outputs/rdt2-finetuned/checkpoint-5000/scheduler.pt
  └─ 0.00 MB
  /home/rishabh/Downloads/umi-pipeline-training/outputs/rdt2-finetuned/checkpoint-4600/optimizer.pt
  └─ 334.58 MB
  /home/rishabh/Downloads/umi-pipeline-training/outputs/rdt2-finetuned/checkpoint-4600/scheduler.pt
  └─ 0.00 MB
  /home/rishabh/Downloads/umi-pipeline-training/outputs/rdt2-finetuned/checkpoint-4800/optimizer.pt
  └─ 334.58 MB
  /home/rishabh/Downloads/umi-pipeline-training/outputs/rdt2-finetuned/checkpoint-4800/scheduler.pt
  └─ 0.00 MB

🔍 Searching for anything with 'normaliz' in name...
  FOUND: /home/rishabh/Downloads/umi-pipeline-training/RDT2/C:\Users\YourName\datasets\normalizer.pt 

In [16]:
# CELL 4.8 - Fix Normalizer Format (params_dict must be nn.ParameterDict)
import os, torch, numpy as np, sys

BASE_DIR        = "/home/rishabh/Downloads/umi-pipeline-training"
RDT2_DIR        = f"{BASE_DIR}/RDT2"
FILTERED_DIR    = f"{BASE_DIR}/shards_marker_only"
NORMALIZER_PATH = f"{BASE_DIR}/umi_normalizer_official.pt"

sys.path.insert(0, RDT2_DIR)
from models.normalizer.normalizer import LinearNormalizer, SingleFieldLinearNormalizer

# act_mean, act_min, act_max, act_std still in memory from previous cell
# If kernel restarted, redefine them here:
# (they're still alive since the cell partially succeeded)

print("🔧 Building normalizer with correct nn.ParameterDict structure...")
print(f"   Action dim: {act_mean.shape[0]}")
print(f"   act_min[:3]:  {act_min[:3]}")
print(f"   act_max[:3]:  {act_max[:3]}")

# ── Build using SingleFieldLinearNormalizer.create_manual ────
# This is the CORRECT way — uses the class's own constructor
# which builds proper nn.ParameterDict internally

scale  = (2.0 / (act_max - act_min + 1e-6)).astype(np.float32)
offset = (-1.0 - scale * act_min).astype(np.float32)

single_field = SingleFieldLinearNormalizer.create_manual(
    scale  = torch.from_numpy(scale),
    offset = torch.from_numpy(offset),
    input_stats_dict = {
        "min":  torch.from_numpy(act_min),
        "max":  torch.from_numpy(act_max),
        "mean": torch.from_numpy(act_mean),
        "std":  torch.from_numpy(act_std),
    }
)

# ── Wrap in LinearNormalizer ──────────────────────────────────
normalizer = LinearNormalizer()
normalizer["action"] = single_field   # uses __setitem__ → correct format

# ── Save ──────────────────────────────────────────────────────
normalizer.save(NORMALIZER_PATH)
size_mb = os.path.getsize(NORMALIZER_PATH) / 1e6
print(f"\n✅ Saved: {NORMALIZER_PATH}  ({size_mb:.3f} MB)")

# ── Verify: load + round-trip ─────────────────────────────────
print("\n🔍 Verifying load + normalize round-trip...")
normalizer2 = LinearNormalizer.load(NORMALIZER_PATH)

test_in   = torch.from_numpy(act_mean).unsqueeze(0)        # (1, 7)
test_out  = normalizer2["action"].normalize(test_in)       # should be ~0
test_back = normalizer2["action"].unnormalize(test_out)    # should be ~act_mean
round_err = (test_back - test_in).abs().max().item()

print(f"   Input  (mean): {test_in[0].tolist()}")
print(f"   Normalized:    {test_out[0].tolist()}")
print(f"   Recovered:     {test_back[0].tolist()}")
print(f"   Round-trip err: {round_err:.2e}")

assert round_err < 1e-4, f"Round-trip error too large: {round_err}"
print(f"\n✅ Normalizer verified!")

# ── Update config ──────────────────────────────────────────────
config = f"""name: custom/robot_dataset
type: single
shards_dir: {FILTERED_DIR}
kwargs:
  instruction_path: {FILTERED_DIR}/instructions.json
  normalizer_path: {NORMALIZER_PATH}
"""
config_path = f"{RDT2_DIR}/configs/datasets/marker_only_dataset.yaml"
with open(config_path, 'w') as f:
    f.write(config)

print(f"✅ Config updated: {config_path}")
print("\n" + "=" * 60)
print("🚀 All set — run Cell 5 to start training!")
print("=" * 60)

🔧 Building normalizer with correct nn.ParameterDict structure...
   Action dim: 7
   act_min[:3]:  [-0.76613414 -0.87396324 -0.03925809]
   act_max[:3]:  [0.3857767  0.29100588 0.37830365]
LinearNormalizer saved to /home/rishabh/Downloads/umi-pipeline-training/umi_normalizer_official.pt

✅ Saved: /home/rishabh/Downloads/umi-pipeline-training/umi_normalizer_official.pt  (0.004 MB)

🔍 Verifying load + normalize round-trip...
LinearNormalizer loaded from /home/rishabh/Downloads/umi-pipeline-training/umi_normalizer_official.pt
   Input  (mean): [0.00891656894236803, -0.09786838293075562, 0.120915487408638, -2.14813494682312, -0.1720350682735443, 0.04704310745000839, 0.026005610823631287]
   Normalized:    [0.3456772565841675, 0.3323858976364136, -0.23281681537628174, -0.6839929223060608, -0.054909609258174896, -0.11493711173534393, 0.04584360122680664]
   Recovered:     [0.008916576392948627, -0.09786837548017502, 0.1209154799580574, -2.14813494682312, -0.1720350682735443, 0.04704310372471

In [ ]:
# CELL 5.1 - Install tensorboard + restart training
import subprocess, sys, os, torch

# ── Install tensorboard ───────────────────────────────────────
print("📦 Installing tensorboard...")
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'tensorboard', 'tensorboardX',
    '--break-system-packages', '-q'
], check=True)
print("✅ tensorboard installed!")

# ── Verify ───────────────────────────────────────────────────
result = subprocess.run(
    [sys.executable, '-c', 'import tensorboard; print(tensorboard.__version__)'],
    capture_output=True, text=True
)
print(f"✅ tensorboard version: {result.stdout.strip()}")

# ── Re-run training ───────────────────────────────────────────
BASE_DIR    = "/home/rishabh/Downloads/umi-pipeline-training"
RDT2_DIR    = f"{BASE_DIR}/RDT2"
OUTPUT_DIR  = f"{BASE_DIR}/outputs/rdt2-marker-only"
DATASET_CFG = "configs/datasets/marker_only_dataset.yaml"
LOG_PATH    = f"{BASE_DIR}/train_marker_only.log"

os.chdir(RDT2_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)

os.environ['TRANSFORMERS_ATTN_IMPLEMENTATION'] = 'eager'
os.environ['PYTORCH_CUDA_ALLOC_CONF']          = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM']           = 'false'

torch.cuda.empty_cache()
free_gb = torch.cuda.mem_get_info(0)[0] / 1e9
print(f"✅ GPU free: {free_gb:.1f}GB")

cmd = [
    sys.executable, 'main.py',

    '--tokenizer_name',                'Qwen/Qwen2.5-VL-7B-Instruct',
    '--vae_name',                      'robotics-diffusion-transformer/RVQActionTokenizer',
    '--pretrained_model_name_or_path', 'robotics-diffusion-transformer/RDT2-VQ',
    '--output_dir',                     OUTPUT_DIR,

    '--train_batch_size',              '2',
    '--gradient_accumulation_steps',   '8',
    '--eval_batch_size',               '8',
    '--max_train_steps',               '3000',
    '--eval_strategy',                 'no',

    '--logging_steps',                 '25',
    '--checkpoints_total_limit',       '6',
    '--checkpointing_step',            '500',

    '--lr_scheduler',                  'cosine',
    '--learning_rate',                 '1e-6',
    '--lr_warmup_steps',               '300',
    '--mixed_precision',               'bf16',
    '--dataloader_num_workers',        '4',
    '--gradient_checkpointing',

    '--log_level',                     'info',
    '--report_to',                     'tensorboard',  # ← now installed
    '--dataset',                        DATASET_CFG,
    '--image_corruption',
    '--use_default_collate_fn_for_eval',

    '--use_qlora',
    '--lora_r',                        '16',
    '--lora_alpha',                    '64',
    '--lora_dropout',                  '0.1',
]

print("\n" + "=" * 60)
print("🚀 LAUNCHING TRAINING")
print("=" * 60)
print(f"GPU:      {free_gb:.1f}GB free")
print(f"Steps:    3000  |  LR: 1e-5  |  LoRA rank: 64")
print(f"Batch:    8 × 8 = 64 effective")
print(f"Log:      {LOG_PATH}")
print("=" * 60)

with open(LOG_PATH, 'w') as logf:
    logf.write("RDT2 Marker-Only Training\n")
    logf.write("=" * 60 + "\n\n")
    logf.flush()

    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=os.environ.copy()
    )

    for line in process.stdout:
        logf.write(line)
        logf.flush()
        print(line, end='', flush=True)

process.wait()

if process.returncode == 0:
    print("\n" + "=" * 60)
    print("✅ TRAINING COMPLETE!")
    print("=" * 60)
    print(f"Checkpoints saved to: {OUTPUT_DIR}")
    print("\nEval order:")
    print("  1. checkpoint-1500  ← start here")
    print("  2. checkpoint-2000  ← likely best")
    print("  3. checkpoint-2500")
    print("  4. checkpoint-3000")
else:
    print(f"\n❌ Failed: exit code {process.returncode}")
    print(f"Check log: {LOG_PATH}")
    # Print last 20 lines of log for quick diagnosis
    print("\n--- Last 20 log lines ---")
    with open(LOG_PATH) as f:
        lines = f.readlines()
        for l in lines[-20:]:
            print(l, end='')

📦 Installing tensorboard...
✅ tensorboard installed!
✅ tensorboard version: 2.20.0
✅ GPU free: 24.8GB

🚀 LAUNCHING TRAINING
GPU:      24.8GB free
Steps:    3000  |  LR: 1e-5  |  LoRA rank: 64
Batch:    8 × 8 = 64 effective
Log:      /home/rishabh/Downloads/umi-pipeline-training/train_marker_only.log

Loading checkpoint shards: 100%|██████████| 4/4 [00:07<00:00,  1.76s/it]
Trainable parameters: 49239808
LinearNormalizer loaded from /home/rishabh/Downloads/umi-pipeline-training/umi_normalizer_official.pt
max_steps is given, it will override any value given in num_train_epochs
Using auto half precision backend
No label_names provided for model class `PeftModel`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
***** Running training *****
  Num examples = 48,000
  Num Epochs = 9,223,372,036,854,775,807
  Instantaneous batch size per device = 2
  

In [10]:
# CELL 4.6 - Inspect Shard Contents
import os, tarfile

FILTERED_DIR = "/home/rishabh/Downloads/umi-pipeline-training/shards_marker_only"

# Look inside first 3 shards to see exact file structure
shard_files = sorted([f for f in os.listdir(FILTERED_DIR) if f.endswith('.tar')])

print("🔍 Inspecting shard contents...")
print("=" * 60)

for shard_name in shard_files[:3]:
    shard_path = os.path.join(FILTERED_DIR, shard_name)
    print(f"\n📦 {shard_name}:")
    with tarfile.open(shard_path, 'r') as tar:
        members = tar.getnames()
        for m in members[:20]:  # show first 20 files
            print(f"   {m}")
        if len(members) > 20:
            print(f"   ... ({len(members)} total files)")
    print("-" * 40)

🔍 Inspecting shard contents...

📦 shard-000000.tar:
   000000.action.npy
   000000.action_token.npy
   000000.image.jpg
   000000.meta.json
   000001.action.npy
   000001.action_token.npy
   000001.image.jpg
   000001.meta.json
   000002.action.npy
   000002.action_token.npy
   000002.image.jpg
   000002.meta.json
   000003.action.npy
   000003.action_token.npy
   000003.image.jpg
   000003.meta.json
   000004.action.npy
   000004.action_token.npy
   000004.image.jpg
   000004.meta.json
   ... (400 total files)
----------------------------------------

📦 shard-000001.tar:
   000100.action.npy
   000100.action_token.npy
   000100.image.jpg
   000100.meta.json
   000101.action.npy
   000101.action_token.npy
   000101.image.jpg
   000101.meta.json
   000102.action.npy
   000102.action_token.npy
   000102.image.jpg
   000102.meta.json
   000103.action.npy
   000103.action_token.npy
   000103.image.jpg
   000103.meta.json
   000104.action.npy
   000104.action_token.npy
   000104.image.jpg
 